<a href="https://colab.research.google.com/github/bookworm6/Data-Science-Final-Project/blob/main/FinalProjectNoteBook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#An Analysis of the Balance Between Sexes Across Subjects at Whitman College

## Motivation
I was one of three girls in my high school computer science class, so a gender gap in computer science felt normal (though not good) to me. I expected that I would experience a similar overwhelmingly large gender gap in computer science in college, so I was surprised to discover that, though my computer science classes are still male dominated, the difference at Whitman does not feel as large as it did in high school. I am doing this project because I wanted to know how my experience fit into larger patterns of gender gaps over time. Aditionally, I want to connect gender gaps to larger economic inequities. However, the insitutional data collected by Whitman documents biological sex. This led me to analyze the balance between biological sexes in various departments at Whitman.

## Defining the Term "Sex Gap"
I have data about biological sex, but not gender. So, I am analyzing the blanace between the sexes in subjects in relation to Whitman's overall demographics. I am calling the difference between the balance between the sexes in a subject and in Whitman as a whole, the "sex gap," and I will use this (made up) term thoughout.   

## Questions
* How have Whitman's overall demographics changed overtime?
* In which departments is the balance between the sexes least representative of Whitman's overall demographics?
* How have the gaps in representation of the sexes in various subjects changed overtime?
* How does the representation of sexes in a subject at whitman relate to the national median mid-career salary of people who major in that subject?

## Data Overview
Most of the data used in this project was collected by Whitman's Office of Institutional Research. There are two parts linked by the unique numerical `Fake ID`s of students.

`studentInfo.csv` gives demographic data about every student who graduated in May 2001 or later represented by their Fake Id. It does not show students who graduated earlier than that, students who left the college without graduating, or students who are currently at Whitman but did not graduate.

The categories of the demographic data are quite broad in order to protect the privacy of individual students.
For each of the following categories, only one value can be in a feild:
* Fake ID of the student
* Gender: represented as M or F. The "Gender" category seems to actually mean biological sex.
* Race/Ethnicity: represented as White, Student of Color, International, or Unknown. I assume that all international students are counted as international regardless of their ethnicity.
* Home State: represented as Oregon, Washington, Caligornia, Other State, or International. For each of these categories, only one value is listed.
* Graduation Year

In the following categories, multiple values can inhabit one feild.
* Major Area: can contain Arts and Humanities, Science and Math, Social Science, or Interdiciplinary. Multiple major areas can be listed when students have multiple majors in different areas.  

`studentInfoNoGraduation.csv` gives deomographic information about students who did not graduate. It has the same categories as `studentInfo.csv`, but instead of graduation year, it has "status" which gives information about their student status (Withdrawn, Withdrawn non-degree-seeking, Current Student, or Graduated before fall 2000). Aditionally, instadod Major Area, it gives the major of any students with majors.

`courseStudentPairs.csv` contains a unique pair of student and course for every course that every student takes. It contains following categores
* Fake ID: this is the FakeID of the student, which links each student to their courses.
* Stu-Course ID: a unique ID of each pair of student and course
* Term: The term is listed as yearSemester. Semester can be SP (spring), FA (fall) or SU (summer).
* Subject: The code of the department offering the class
* Section: The course code complete with a section ID if multiple sections of the course were taught the same mester
* Title: the course title
* Credits: the number of credits the student took the class for.

Finally, salary by major information comes from the wall street journal, and is in `salaries by major - salaries by major.csv`. It gives the following self explanatory categories:
* Undergraduate Major
* Starting Median Salary
* Mid-Career Median Salary
* Percent change from Starting to Mid-Career Salary
* Mid-Career 10th Percentile Salary
* Mid-Career 25th Percentile Salary
* Mid-Career 75th Percentile Salary
* Mid-Career 90th Percentile Salary

The salary by major data set had minimal meta data, and was not dated. However, judging by the design of [the website](https://www.wsj.com/public/resources/documents/info-Degrees_that_Pay_you_Back-sort.html), it is probably at least ten years old.

The salary by major data set did not contain all of Whitman's majors and its names did not perfectly align with Whitman's. `salaries by major - course codes to major names.csv` contains my manual (somewhat subjective) mapping of major names onto Whitman's departments.

## Acknowledgements
Thank you to Neal Christopherson in Whitman's Office of Instituional Research for providing me with data and all of the work he put into anonymizing the data.


## Cleaning Data
In order to answer the questions, I first needed to make sense of the structure of the data, clean it, adjust data types, and combine the demographic inormation with the course information

#### Loading and Examining Data
My first step was to import my libraries, load my data into data frames, and examine the data frames and the types of my data.

In [102]:
#importing libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio

#loading data
studentInfo = pd.read_csv("studentInfo.csv")
studentInfoNonGraduated = pd.read_csv("studentInfoNonGraduates.csv")
courseStudentPairs = pd.read_csv("courseStudentPairs.csv")

In [ ]:
studentInfo

In [ ]:
studentInfoNonGraduated

In [ ]:
courseStudentPairs

After viewing my data frames, I concatinated studentInfoNonGraduated and studentInfo. They are so similar that it makes sense to clean them together.

In [106]:
#adding status column to the data frame of graduated students. Since all of them are graduated their status is graduated.
studentInfo["Status"] = "Graduated"

#concatinating studentInfo and studentInfoNonGraduated.
studentInfo = pd.concat([studentInfo,studentInfoNonGraduated],ignore_index=True) #AI overview reminded me of the syntax for concat and told me that concat acted like an outer join and sets values is missing columns to NaN.

In [ ]:
#viewing concatinated data frame
studentInfo

Then, I examined the data types in my data frames to see if I should change them to make analysis easier.

In [108]:
#examing data types of demographic information. None of the data types needed to be changed
studentInfo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12787 entries, 0 to 12786
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Fake ID          12787 non-null  int64  
 1   Graduation Year  9130 non-null   float64
 2   Race/Ethnicity   12786 non-null  object 
 3   Gender           12630 non-null  object 
 4   Major Area       9130 non-null   object 
 5   Home State       12114 non-null  object 
 6   Status           12787 non-null  object 
 7   Majors           3657 non-null   object 
dtypes: float64(1), int64(1), object(6)
memory usage: 799.3+ KB


In [109]:
#examining data types of course student pairs. Term needs to be changed to to a numeric data type so that I can do analysis over time
courseStudentPairs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 409304 entries, 0 to 409303
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   Stu-Course ID  409304 non-null  int64 
 1   Fake ID        409304 non-null  int64 
 2   Term           409304 non-null  object
 3   Subject        409304 non-null  object
 4   Section        409304 non-null  object
 5   Title          409304 non-null  object
 6   Credits        409304 non-null  int64 
dtypes: int64(3), object(4)
memory usage: 21.9+ MB


The only data type that does not work is the data type of Term in couseStudentPairs. A string is not ideal for analysing trends overtime because I want to be able to sort it by numberical value.


#### Determining Possible Categorical Values
In order to analyse categorical data, I needed to understand what categories I had. So, I used the pandases unique() function to get a list of all possible categories in the categorical data that interested me.

To determine how gender was represented, I viewed all of the possible values of gender in the data


In [110]:
#viewing possible values of gender
studentInfo["Gender"].unique() #credit to AI overview for teaching me about the unique() function

array(['F', 'M', '0', nan], dtype=object)

I wasn't sure what 0 meant, so I examined the sole row in this data frame that has 0 as Gender.

In [ ]:
#examing rows with 0 gender
studentInfo[studentInfo["Gender"]=="0"]

Since there was only one data point with 0, I can not draw any conclusions about it.

Next, I tried to figure out what NaN gender meant. I guessed that it probably either meant that the student was nonbinary, or that there was not data on the student's gender. To figure it out, I examined all of the students with NaN gender.

In [ ]:
studentInfo[studentInfo["Gender"].isna()]

I noticed that many of the students with NaN gender appeared to be non-degree seeking. To see if this was a real pattern, I decided to count the number of occurences of each student status in the subset of students with NaN gender.

In [113]:
studentInfo[studentInfo["Gender"].isna()].groupby("Status")["Status"].agg('count')

,Status
Status,
Current Student,5
Non-degree-seeking student,47
Withdrawn,2
Withdrawn non-degree-seeking,103


The vast majority of students with NaN gender were non degree seeking. It seems reasonable that the college would have less data on non degree seeking students, which indicates that NaN probably means the college does not have data on gender.

These "gender" categories are not ideal. Since there was only one 0 and NaN appeared to indicate missing data, I am pretty sure that "gender" actually represents biological sex in this data set. This is supported by names of the categories (F for female and M for male).

Since gender values of 0 and NaN are both unknown, I decided to replace both with "U" for unknown.

In [114]:
studentInfo["Gender"] = studentInfo["Gender"].replace(np.nan,"U") #credit to AI overview for telling me about replace function
studentInfo["Gender"] = studentInfo["Gender"].replace("0","U") #credit to AI overview for telling me about replace function

To determine how race and ethnicity was represented, I viewed all of the possible values of race/ethnicity in the data.


In [115]:
#viewing possible values of race and ethnicity
studentInfo["Race/Ethnicity"].unique()

array(['White', 'Student of Color', 'Unknown', 'International', nan],
      dtype=object)

I examined students with "Unknown" and NaN race to see if I could find patterns in which ones were marked "Unknown" or NaN

In [ ]:
studentInfo[studentInfo["Race/Ethnicity"].isna()]

Only one student had NaN race.

In [ ]:
studentInfo[studentInfo["Race/Ethnicity"]=="Unknown"]

Unknown race was much more common. Since both NaN and Unknown Race/Ethnicity are unknown, I decided to mark them both "Unknown".  

In [118]:
#replacing NaN reace with "Unknown"
studentInfo["Race/Ethnicity"] = studentInfo["Race/Ethnicity"].replace(np.nan,"Unknown")

#### Reformatting and Combining Data for Easier Use


In the original data, terms were formated at strings, which makes it hard to examine trends over time.
I reinterpreted term as a float where the number before the decimal is the year, and the number after the decimal represents the semester

In [119]:


#seeing what semester codes could exist at the end a term
courseStudentPairs["Term"].unique()

array(['2000FA', '2001SP', '2001FA', '2002SP', '2002FA', '2003SP',
       '2003FA', '2004SP', '2004FA', '2005SP', '2005FA', '2006SP',
       '2006FA', '2007SP', '2007FA', '2008FA', '2008SU', '2009SP',
       '2009FA', '2010SP', '2010FA', '2011SP', '2011FA', '2012SP',
       '2012FA', '2013SP', '2013FA', '2014SP', '2014FA', '2015SP',
       '2015FA', '2016SP', '2016FA', '2017SP', '2017FA', '2018SP',
       '2018FA', '2019SP', '2019FA', '2020SP', '2020FA', '2021SP',
       '2021FA', '2022SP', '2022FA', '2023SP', '2023FA', '2024SP',
       '2024FA', '2025SP', '2025FA', '2026SP'], dtype=object)

In [120]:
#making term year and semester columns, and expressing term numerically.
#term year
courseStudentPairs["TermYear"] = courseStudentPairs["Term"].apply(lambda x: int(x[0:len(x)-2]))

#term semester
courseStudentPairs["TermSemester"]= courseStudentPairs["Term"].apply(lambda x:x[len(x)-2:])

#make term a float (number before decimal is year, number after decimal is semster)
semesterToFloat = {"FA":0.5, "SU":0.4,"SP":0}
courseStudentPairs["Term"] = courseStudentPairs["Term"].apply(lambda x: float(x[0:len(x)-2])+semesterToFloat[x[len(x)-2:]])

I want demographic information about every student taking every course, So I merged the student information into courseStudentPairs. Having this information in one data frame makes analysis easier


In [121]:
#I want demographic information about every student taking every course, So I merged the student information into courseStudentPairs

#joining student information into courseStudentPairs on Fake ID
df = courseStudentPairs.merge(studentInfo,on="Fake ID",how = "left")

In [ ]:
#examining the cleaned up data frame
df


## Question 1: What are Whitman's general demographic trends overtime?

Before analyzing the sex gaps in subjects overtime. I wanted to look at Whitman's overall demographic trends. This is important context for any analysis of sex gaps.

I calculated the percentage of class seats filled by people of each demographic in every term, and graphed it. I chose to base these demographics off of class seats instead of enrolled students because some students take more classes than others and I wanted the demographic analysis to reflect the demographic make up of an average class.

In [123]:
#creating demographics, which gives the number of course student pairs with each demographic each term. in other words, the number of class seats filled by people of each demographic
demographics = df.groupby(["Term","Race/Ethnicity","Gender"])["Gender"].count().fillna(0).reset_index(name='count') #credit to gemmini for telling me i could reset the index to get rid of the weird nesting things
#adding totals column, which contains the total number of students in each graduating class
demographics["totals"] = demographics.groupby("Term")["count"].transform('sum') #credit to gemmini for telling me about the transform function

#calculating the percent of class seats filled by each population
demographics["percent"] = demographics["count"]/demographics["totals"]*100

#calculating statistics of class seats filled by each sex without regard to race
demographicsNoRace = demographics.groupby(["Term","Gender"]).agg({'percent':'sum','count':'sum',"totals":"mean"}).reset_index()
#adding the totals to the main demographic data frame
demographicsNoRace["Race/Ethnicity"]="all"
demographics = pd.concat([demographicsNoRace,demographics],ignore_index=True)




Then, I graphed these percentages overtime, excluding the unknown categories.


In [124]:
#excluding groups with unknown race or gender
d = demographics[(demographics["Race/Ethnicity"]!="Unknown")&(demographics["Gender"]!="U")]
#graphing!
px.line(d,x="Term",y="percent",color="Gender",line_dash = "Race/Ethnicity",title="Whitman Demographics Over Time",labels = {"Gender":"Sex","percent":"Percent of Class Seats Filled by Population"})


The summer of 2008 apears to break demographic patterns. I was surprised to learn that Whitman offered summer classes, so I examined the course student pairs from this summer

In [ ]:
#viewing the course student pairs
courseStudentPairs[courseStudentPairs["Term"]==2008.4]

From the class names, it looked like there was a chinese program and an anthropology program running that summer. I wanted to know how many unique students in total were in both programs combined.

In [126]:
print(f"there were {len(courseStudentPairs[courseStudentPairs["Term"]==2008.4]["Fake ID"].unique())} students")

there were 26 students


In total, only 26 students took classes the summer of 2008. This likely skewed demographic data just through randomness. So, I decided to exclude the summer of 2008 from all future analysis.

In [127]:
#removing course student pairs from the summer of 2008
courseStudentPairs = courseStudentPairs[courseStudentPairs["Term"]!=2008.4]

#removing demographic data from the summer of 2008
demographics = demographics[demographics["Term"]!=2008.4]

Then, I graphed the demographics again

In [128]:
#excluding groups with unknown race or gender
d = demographics[(demographics["Race/Ethnicity"]!="Unknown")&(demographics["Gender"]!="U")]
#graphing
demographicsGraph = px.line(d,x="Term",y="percent",color="Gender",line_dash = "Race/Ethnicity",title="Whitman Demographics Over Time",labels = {"Gender":"Sex","percent":"Percent of Class Seats Filled by Population"})
demographicsGraph.write_html("demographics.html")


Based on this graph, I can now describe Whitman's demographic trends overtime

* Across all Race/Ethnicity categories except International, Women tend to make up a larger percentage of the population then Men.
* International students are much more evenly balanced by sex.
* Most students at Whitman are white, but the percentage of white students trends down overtime while the percentage of students of color and international students trends slightly up.

## Question 2: Which departments have the largest gender gaps on average?


My metric to measure demmographic gaps is "percentDifference," which is the difference between the percentage of class seats in a subject filled by a demographic and the percentage of class seats filled throughout the school by that sex. Positive percent_differences are the percentage of seats in the class filled by the demographic that would not be filled by that demographic in a class representative of Whitman's population. Negative percentDifference in a class is the percentage of class seats that are not filled by that demographic that would be in a a class representative of Whitman's population.

In [129]:
#calculating percent_subject, which is the percentage of class seats in a subject filled by a sex race pair .

#creating demographics by subject with a count column which is number of people in a subject with a gender and race in a term
demographicsBySubject = df.groupby(["Term","Subject","Gender","Race/Ethnicity"])["Gender"].count().fillna(0).reset_index(name='count')

#adding subject_totals which are number of people in a subject in a term
demographicsBySubject["subject_totals"] = demographicsBySubject.groupby(["Term","Subject"])["count"].transform('sum') #credit to gemmini for telling me about the transform function

#adding a race "all" which is the statistics for all of the races
tempDNoRace = demographicsBySubject.groupby(["Term","Subject","Gender"]).agg({'count':'sum',"subject_totals":"mean"}).reset_index()
tempDNoRace["Race/Ethnicity"]="all"
demographicsBySubject = pd.concat([tempDNoRace,demographicsBySubject],ignore_index=True)

#calculating percentage of subject in a term filled by people of a demographic
demographicsBySubject["percent_subject"] = demographicsBySubject["count"]/demographicsBySubject["subject_totals"]*100

demographicsBySubject


,Term,Subject,Gender,count,subject_totals,Race/Ethnicity,percent_subject
0,2000.5,ANTH,F,50,90.0,all,55.555556
1,2000.5,ANTH,M,40,90.0,all,44.444444
2,2000.5,ART,F,222,335.0,all,66.268657
3,2000.5,ART,M,113,335.0,all,33.731343
4,2000.5,ASNS,F,13,24.0,all,54.166667
...,...,...,...,...,...,...,...
18628,2026.0,THDN,F,38,204.0,Student of Color,18.627451
18629,2026.0,THDN,F,105,204.0,White,51.470588
18630,2026.0,THDN,M,13,204.0,International,6.372549
18631,2026.0,THDN,M,10,204.0,Student of Color,4.901961


In [130]:
#removing subjects by term in which fewer than 10 people took classes in that subject in a term. because there is too much space for randomness
demographicsBySubject = demographicsBySubject[demographicsBySubject["subject_totals"]>=10]

#adding a column percent_wholeSchool, which is is the percent of seats filled throughout the school by a sex race pair
demographicsBySubject = demographicsBySubject.merge(demographics[["Term","Gender","Race/Ethnicity","percent"]],how = "left",on = ["Term","Gender","Race/Ethnicity"])
demographicsBySubject = demographicsBySubject.rename(columns = {"percent":"percent_wholeSchool"}) #credit to AI overview for telling me how to rename columns

#calculating percentDifference, which is the difference between the percentage of class seats in a subject filled by a sex race pair and the percentage of class seats filled throughout the school by that same pair
#this is the metric I will use to measure subject dependent sex gaps.
demographicsBySubject["percentDifference"] = demographicsBySubject["percent_subject"] - demographicsBySubject["percent_wholeSchool"]

demographicsBySubject.to_csv("demographicsBySubject.csv")

In [131]:
demographicsBySubject

,Term,Subject,Gender,count,subject_totals,Race/Ethnicity,percent_subject,percent_wholeSchool,percentDifference
0,2000.5,ANTH,F,50,90.0,all,55.555556,57.177835,-1.622279
1,2000.5,ANTH,M,40,90.0,all,44.444444,42.822165,1.622279
2,2000.5,ART,F,222,335.0,all,66.268657,57.177835,9.090822
3,2000.5,ART,M,113,335.0,all,33.731343,42.822165,-9.090822
4,2000.5,ASNS,F,13,24.0,all,54.166667,57.177835,-3.011168
...,...,...,...,...,...,...,...,...,...
18030,2026.0,THDN,F,38,204.0,Student of Color,18.627451,18.951291,-0.323840
18031,2026.0,THDN,F,105,204.0,White,51.470588,33.058291,18.412297
18032,2026.0,THDN,M,13,204.0,International,6.372549,6.454618,-0.082069
18033,2026.0,THDN,M,10,204.0,Student of Color,4.901961,10.420548,-5.518588


I wanted to be able to visualize the distribution of gaps between sexes across subjects, and I wanted to be able to explore each point and see which subjects they coorespond to. So, I visualized percentDifference on a numberline where each point represents the average sex gap in that subject. Hovering over the point gives the subject.

In [132]:
#grouping by subject and gender where race/Ethnicity is all and averaging percentDifference
averageDifferences = demographicsBySubject[demographicsBySubject["Race/Ethnicity"]=="all"].groupby(["Subject","Gender"])["percentDifference"].mean().reset_index()


#adding y values to divide scatter plot into 3 number lines
genderY = {"F":1,"M":0,"U":-1}
averageDifferences["y"] = averageDifferences['Gender'].apply(lambda x: genderY[x])

#plotting
#https://plotly.com/python-api-reference/generated/plotly.express.scatter
averageDiffNumberline = px.scatter(averageDifferences,"percentDifference","y",opacity=0.3,hover_name="Subject",color="Gender",hover_data={"y":False,"Gender":False,"percentDifference":False},title="Average Over/Under Representation of the Sexes across Subjects",labels={"Gender":"Sex","percentDifference":"percent under represented                          percent over represnted"}) #credit to AI overview for teaching me how to control hover data and labels
averageDiffNumberline.update_yaxes(visible=False) #credit to AI overview



This graph shows that female students are most underrepresented, on average, in CS and econ, and male students are generally most underrepresented in social justice and dance.

## Question 3: How have the gaps in representation of the sexes in various subjects changed overtime?


In order to explore this question, I expanded on the visualization above to create a tool where, when you click on a point, it shows a graph of of the representation of the sexes in that major overtime. Aditionally, though these lines are not shown by default, the over/under representation of sex race pairs can be shown by clicking on that sex race pair in the key.

I decided to make this an interactive visualization in the hopes that people would slow down and explore. I wanted people to consider majors individually without being overwhelmed.

I wanted to show data about both sexes so that the audience could get a complete story in which, when one sex was over represented, the other was under represented.

In [134]:
#installing dash, which is a library for interactive visualizaions (credit to gemmini)
!pip install dash
!pip install dash comm

In [135]:
#importing functions
from dash import Dash, dcc, html, Input, Output


In [136]:
#Sources for this section. I learned how dash worked and how to do callbacks using Datacamp. https://app.datacamp.com/learn/courses/building-dashboards-with-dash-and-plotly
#Then I asked Gemmini questions and had it help me figure out get and parse clickData.
#Gemmini also helped me trouble shoot displaying the dashboard in the notebook (It turns out I had to install dash comm)


#creating dash app
app = Dash(__name__)
#setting layout. including the numberline graph above and a blank graph to be filled
app.layout = html.Div([
    dcc.Graph( id = "numberline",figure = averageDiffNumberline),
    dcc.Graph(id = "zoomin")
])

#creating callback
@app.callback(
    Output("zoomin","figure"),
    Input("numberline","clickData")
)
#defining update function
def update(clickData):
  if clickData is None:
    return px.line(title = "please click a point")
  subject = clickData["points"][0]["hovertext"]
  subjectDemographics = demographicsBySubject[demographicsBySubject["Subject"]==subject]
  graph = px.line(subjectDemographics, x="Term",y="percentDifference",color = "Gender", line_dash="Race/Ethnicity",title = f"Representation of the Sexes in {subject} over time",labels={"percentDifference":"percent under/over represented","Gender":"Sex"})
  for trace in graph.data: #credit to gemmini for helping me hide traces
    if "all" not in trace.name:
      trace.update(visible="legendonly")
  return graph

#running app
if __name__ == '__main__':
    app.run(jupyter_mode="inline")

<IPython.core.display.Javascript object>

After exploring the visualizaton above, I decided to
highlight English and CS. I wanted to highlight CS because my audience for this project is mostly CS majors and CS has a large sex gap. I wanted to highlight English because it suddenly changes from a major with a pretty even distribution across sexes to a major with a large sex gap.

In [137]:
#graphing representation in CS over time
subject = "CS"
subjectDemographics = demographicsBySubject[demographicsBySubject["Subject"]==subject]
graph = px.line(subjectDemographics, x="Term",y="percentDifference",color = "Gender", line_dash="Race/Ethnicity",title = f"Representation of the Sexes in {subject} over time",labels={"percentDifference":"percent under/over represented","Gender":"Sex"})
for trace in graph.data: #credit to gemmini for helping me hide traces
  if "all" not in trace.name:
    trace.update(visible="legendonly")
graph.add_annotation( ax=0, ay=100, x=2022.5,y=23.7,text="Chat GPT Launched. Beginning of Job Market Decline for New Grads", showarrow=True, arrowhead = 3) #credit to AI overview for telling me how to do annotations
graph.add_annotation( ax=0, ay=-110, x=2022.5,y=-23.4, showarrow=True, arrowhead = 3)

graph.write_html("CS Over Time.html")
graph

On average, females are most underrepresented in CS and males are most over represented. This gap was fairly constant through 2021, but then it started to narrow very slowly in the spring of 2022.

Interestingly, the fall of was the year that ChatGPT launched, which [was the beginning of a 20% decrease in employment for new CS grads. ](https://digitaleconomy.stanford.edu/app/uploads/2025/11/CanariesintheCoalMine_Nov25.pdf). It is unclear how this effected the rate that the cap closed at.

In [138]:
#graphing representation in english over time
subject = "ENGL"
subjectDemographics = demographicsBySubject[demographicsBySubject["Subject"]==subject]
graph = px.line(subjectDemographics, x="Term",y="percentDifference",color = "Gender", line_dash="Race/Ethnicity",title = f"Representation of the Sexes in English (ENGL) over time",labels={"percentDifference":"percent under/over represented","Gender":"Sex"})
for trace in graph.data: #credit to gemmini for helping me hide traces
  if "all" not in trace.name:
    trace.update(visible="legendonly")

graph.write_html("ENGL Over Time.html")
graph

Female and male students were fairly evenly represented in English classes until 2014. In 2014, there started to be a clear gap in which female students were over represented and male students were underrepresented. This is interesting because a sudden change probably has a very specific cause, but I have not been able to speculate what that cause was.

## Question 4: How does the Representation of Sexes in a Major At Whitman Relate to The National Median Mid-Career Salary of That Major?

I needed data about salaries by major, which I obtained from the wallstreet journal. https://www.wsj.com/public/resources/documents/info-Degrees_that_Pay_you_Back-sort.html

This data set used complete names of majors, which I manually mapped to whitman subject/course codes to the best of my ability. There is some amount of subjectivity in this mapping. Unfortunately, I do not have salary data for all of Whitman's majors, so when I merged salary data into the data on representation of sexes, I excluded majors that I did not have salary data for.

In [139]:
#loading key linking course/subject codes to the major. names in the salary data
courseCodesToNames = pd.read_csv("salaries by major - course codes to major names.csv")
courseCodesToNames = courseCodesToNames[["Course Code", "Major Name"]]
#loading national salary data by major
salaryDf = pd.read_csv("salaries by major - salaries by major.csv")

#using the key dataframe (courseCodesToNames)to merge percent over/under represented into salary data
salaryDf = salaryDf.merge(courseCodesToNames,how="inner",left_on="Undergraduate Major",right_on = "Major Name")
salaryDf = salaryDf.merge(averageDifferences,how="inner",left_on = "Course Code",right_on = "Subject")

#removing unecessary columns for clarity
salaryDf = salaryDf[["Major Name", "Course Code", "Mid-Career Median Salary", "Starting Median Salary","percentDifference","Gender"]]

In [140]:
#examining merged data frame
salaryDf

,Major Name,Course Code,Mid-Career Median Salary,Starting Median Salary,percentDifference,Gender
0,Anthropology,ANTH,"$61,500.00","$36,800.00",8.218909,F
1,Anthropology,ANTH,"$61,500.00","$36,800.00",-8.134663,M
2,Anthropology,ANTH,"$61,500.00","$36,800.00",0.622926,U
3,Art History,ARTH,"$64,900.00","$35,800.00",8.877526,F
4,Art History,ARTH,"$64,900.00","$35,800.00",-8.775368,M
...,...,...,...,...,...,...
64,Spanish,SPAN,"$53,100.00","$34,000.00",15.189794,F
65,Spanish,SPAN,"$53,100.00","$34,000.00",-15.169307,M
66,Spanish,HISP,"$53,100.00","$34,000.00",7.541987,F
67,Spanish,HISP,"$53,100.00","$34,000.00",-7.505789,M


In [141]:
#inspecting types in the new data frame
salaryDf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69 entries, 0 to 68
Data columns (total 6 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Major Name                69 non-null     object 
 1   Course Code               69 non-null     object 
 2   Mid-Career Median Salary  69 non-null     object 
 3   Starting Median Salary    69 non-null     object 
 4   percentDifference         69 non-null     float64
 5   Gender                    69 non-null     object 
dtypes: float64(1), object(5)
memory usage: 3.4+ KB


In [142]:
#the $ and , in the salary information caused salary information to be interpreted as a string. I reinterpreted it as a float
salaryDf["Mid-Career Median Salary"] = salaryDf["Mid-Career Median Salary"].apply(lambda x: float("".join(x[1:].split(","))))

Finally, I graphed the median mid career salary of each major against the representation of each sex in each major. I decided to make two seperate graphs for clarity.

In [143]:
#graphing salary vs representation for Females
salaryDfWomen = salaryDf[salaryDf["Gender"]=="F"]
salaryGraph = px.scatter(salaryDfWomen,x="percentDifference",y="Mid-Career Median Salary",title="National Mid-Career Median Salary Vs Representation of Female Students At Whitman",hover_name="Course Code",hover_data={"percentDifference":False,"Gender":False,"Mid-Career Median Salary":False},labels={"percentDifference":"average percent under represented                          average percent over represnted","Gender":"Sex","Mid-Career Median Salary":"Mid-Career Median Salary ($)"})
salaryGraph.write_html("SalaryGraphFemale.html")
salaryGraph

In [144]:
#graphing salary vs representation for Females
salaryDfMale = salaryDf[salaryDf["Gender"]=="M"]
salaryGraph = px.scatter(salaryDfMale,x="percentDifference",y="Mid-Career Median Salary",title="National Mid-Career Median Salary Vs Representation of Male Students At Whitman",hover_name="Course Code",hover_data={"percentDifference":False,"Gender":False,"Mid-Career Median Salary":False},labels={"percentDifference":"average percent under represented                          average percent over represnted","Gender":"Sex","Mid-Career Median Salary":"Mid-Career Median Salary ($)"})
salaryGraph.write_html("SalaryGraphMale.html")
salaryGraph


Female students at Whitman are underrepresented in subjects, nationally, lead to higher paying careers. Male students are over represented in these subjects



.

##Sources
* https://digitaleconomy.stanford.edu/app/uploads/2025/11/CanariesintheCoalMine_Nov25.pdf
* https://www.wsj.com/public/resources/documents/info-Degrees_that_Pay_you_Back-sort.html
* https://app.datacamp.com/learn/courses/building-dashboards-with-dash-and-plotly
* Whitman's Instituional Research Data
* Gemmini (see comments)
* Google AI Overview (see comments)


### A Note On AI Use
In this project, I used Google's AI Overview for small things as an alternative to reading documentation. I have used this method in the past and I find it helpful because it saves me time without circumventing my learning

I learned how to use Dash using datacamp, but I used Gemmini to clarify points, give details that datacamp did not, and troubleshoot my interactive visualizations. I don't think that Gemmini circumvented this part of my learning. I also used Gemmini to figure out how to host my interactive Dashboards. This was mostly a good idea because I was able to talk through pros and cons of different methods with Gemmini, which prevented me from wasting time. When I did decide to use Render, Gemmini was able to describe how to set it up well. Since Gemmini helped me avoid mistakes while setting up Render, Gemmini may have prevented me from learning as much about using Render as I otherwise would have. However, I intentionally made a choice not to prioritize learning about Render, which I probably won't use again, and I am okay with that choice.  

### Reflection
I was most negatively surprised by the realization that GitHub pages can not host interactive python apps. I had finished my interactive visualization, and I was quite proud of it. I thought I was done. I googled "dash app to html" expecting to find a function similar to the one for creating html plotly graphs, and I quickly learned that there was not an easy solution. I am extremely glad that hosting services such as Render exist, but I know that I got really lucky. In the future, I will plan out how I will encorporate a tool fully before I start using it.

The most challenging part was figuring out how to present my analysis in an engageing way without loosing any of the nuance. I tried to balance my desire to make all of my decisions transparent against an understanding my various disclaimers are probably tedious for my audience. I am still not sure if I struck the right balance.